# Estudo de Caso 8.2 — Estabilidade Numérica: Diferenças Finitas

**Capítulo Associado:** Capítulo 8 — Transferência de Calor em Solos  
**Nível:** Pós-Graduação  
**Tempo estimado:** 60 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, analisamos a estabilidade de esquemas numéricos para a equação de calor:

1. **Esquema Explícito (Euler para frente):** Condicionalmente estável
2. **Esquema Implícito (Euler para trás):** Incondicionalmente estável
3. **Esquema de Crank-Nicolson:** Incondicionalmente estável e mais preciso

**Objetivo:** Comparar os três esquemas e identificar violações da condição CFL.

---

## 🎯 Objetivos de Aprendizagem

- Implementar os três esquemas numéricos
- Analisar a condição de estabilidade (número de Fourier)
- Visualizar instabilidades numéricas
- Comparar precisão e custo computacional

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `matplotlib`
- Conhecimento prévio: Esquemas numéricos (Seção 8.4)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Ambiente configurado com sucesso!")

---

## 🧮 Seção 1: Definição dos Esquemas

### Esquema Explícito (Euler para Frente)

$$
T_i^{n+1} = T_i^n + r \cdot (T_{i+1}^n - 2T_i^n + T_{i-1}^n)
$$

**Condição de estabilidade:** $r \leq 0,5$

### Esquema Implícito (Euler para Trás)

$$
-r \cdot T_{i-1}^{n+1} + (1 + 2r) \cdot T_i^{n+1} - r \cdot T_{i+1}^{n+1} = T_i^n
$$

**Condição de estabilidade:** Incondicionalmente estável

### Esquema de Crank-Nicolson

$$
-\frac{r}{2} T_{i-1}^{n+1} + (1 + r) T_i^{n+1} - \frac{r}{2} T_{i+1}^{n+1} = \frac{r}{2} T_{i-1}^n + (1 - r) T_i^n + \frac{r}{2} T_{i+1}^n
$$

**Condição de estabilidade:** Incondicionalmente estável (mas pode oscilar se $r > 1$)

In [ ]:
# ============================================================
# IMPLEMENTAÇÃO DOS ESQUEMAS
# ============================================================

def thomas_algorithm(a, b, c, d):
    """Resolve sistema tridiagonal Ax = d."""
    n = len(d)
    c_ = np.zeros(n)
    d_ = np.zeros(n)
    
    c_[0] = c[0] / b[0]
    d_[0] = d[0] / b[0]
    for i in range(1, n):
        denom = b[i] - a[i] * c_[i - 1]
        c_[i] = c[i] / denom
        d_[i] = (d[i] - a[i] * d_[i - 1]) / denom
    
    x = np.zeros(n)
    x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    
    return x

def esquema_explicito(T, r):
    """Esquema explícito (Euler para frente)."""
    T_new = T.copy()
    for i in range(1, len(T) - 1):
        T_new[i] = T[i] + r * (T[i+1] - 2*T[i] + T[i-1])
    return T_new

def esquema_implicito(T, r):
    """Esquema implícito (Euler para trás)."""
    N = len(T)
    a = -r * np.ones(N)
    b = (1 + 2*r) * np.ones(N)
    c = -r * np.ones(N)
    d = T.copy()
    
    # Condições de contorno
    d[0] += r * T[0]
    d[-1] += r * T[-1]
    
    return thomas_algorithm(a, b, c, d)

def esquema_crank_nicolson(T, r):
    """Esquema de Crank-Nicolson."""
    N = len(T)
    a = -(r/2) * np.ones(N)
    b = (1 + r) * np.ones(N)
    c = -(r/2) * np.ones(N)
    
    d = np.zeros(N)
    for i in range(1, N - 1):
        d[i] = (r/2)*T[i-1] + (1-r)*T[i] + (r/2)*T[i+1]
    d[0] = T[0]
    d[-1] = T[-1]
    
    # Condições de contorno
    d[0] += (r/2) * T[0]
    d[-1] += (r/2) * T[-1]
    
    return thomas_algorithm(a, b, c, d)

print("✅ Esquemas implementados!")

---

## 🧪 Seção 2: Teste de Estabilidade

Vamos testar os esquemas com diferentes valores de $r$ para identificar instabilidades.

In [ ]:
# ============================================================
# TESTE DE ESTABILIDADE
# ============================================================

print("=" * 60)
print("TESTE DE ESTABILIDADE - VARIAÇÃO DE r")
print("=" * 60)

# Parâmetros
N = 51
L = 0.5
z = np.linspace(0, L, N)
N_steps = 100

# Condição inicial: degrau de temperatura
T_init = np.full(N, 20.0)
T_init[N//2:] = 30.0

# Valores de r a testar
r_values = [0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, r in enumerate(r_values):
    ax = axes[idx]
    
    # Esquema explícito
    T_exp = T_init.copy()
    instabilidade_exp = False
    for step in range(N_steps):
        T_exp = esquema_explicito(T_exp, r)
        if np.any(np.abs(T_exp) > 1000):
            instabilidade_exp = True
            break
    
    # Esquema implícito
    T_imp = T_init.copy()
    for step in range(N_steps):
        T_imp = esquema_implicito(T_imp, r)
    
    # Plot
    ax.plot(z * 100, T_init, 'k--', linewidth=1, label='Inicial')
    if not instabilidade_exp:
        ax.plot(z * 100, T_exp, 'b-', linewidth=2, label=f'Explícito (r={r})')
    else:
        ax.text(0.5, 0.5, 'INSTÁVEL!', transform=ax.transAxes,
                fontsize=14, color='red', ha='center', fontweight='bold')
    ax.plot(z * 100, T_imp, 'r-', linewidth=2, label=f'Implícito (r={r})')
    
    ax.set_xlabel('Profundidade [cm]', fontsize=11)
    ax.set_ylabel('Temperatura [°C]', fontsize=11)
    ax.set_title(f'r = {r}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    if r <= 0.5:
        ax.text(0.02, 0.98, '✓ Estável', transform=ax.transAxes,
                fontsize=10, color='green', fontweight='bold')
    else:
        ax.text(0.02, 0.98, '⚠ Explícito instável', transform=ax.transAxes,
                fontsize=10, color='orange', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("  • r ≤ 0,5: ambos os esquemas são estáveis")
print("  • r > 0,5: esquema explícito torna-se INSTÁVEL")
print("  • Esquema implícito permanece estável para qualquer r")
print("  • Para r muito altos, implícito pode ter oscilações (mas não explode)")

---

## 📊 Seção 3: Comparação de Precisão

In [ ]:
# ============================================================
# COMPARAÇÃO DE PRECISÃO
# ============================================================

print("=" * 60)
print("COMPARAÇÃO DE PRECISÃO")
print("=" * 60)

# Solução analítica (para comparação)
def solucao_analitica(z, t, alpha, T_media, A, omega):
    beta = np.sqrt(omega / (2 * alpha))
    return T_media + A * np.exp(-beta * z) * np.sin(omega * t - beta * z)

# Parâmetros
alpha = 8.33e-7    # m²/s
T_media = 25.0     # °C
A = 10.0           # °C
omega = 2 * np.pi / 86400
t_final = 43200    # 12 h

# Discretização
N = 51
L = 0.5
z = np.linspace(0, L, N)
dz = L / (N - 1)

# Testar diferentes Δt
dt_values = [60, 300, 600, 1800]  # 1, 5, 10, 30 min

resultados = {'dt': [], 'r': [], 'erro_exp': [], 'erro_imp': [], 'erro_cn': []}

for dt in dt_values:
    r = alpha * dt / dz**2
    N_steps = int(t_final / dt)
    
    # Condição inicial
    T_init = np.full(N, T_media)
    
    # Esquema explícito (se estável)
    if r <= 0.5:
        T_exp = T_init.copy()
        for step in range(N_steps):
            t = step * dt
            T_exp = esquema_explicito(T_exp, r)
            T_exp[0] = T_media + A * np.sin(omega * (t + dt))
        T_ana = solucao_analitica(z, t_final, alpha, T_media, A, omega)
        erro_exp = np.mean(np.abs(T_exp - T_ana))
    else:
        erro_exp = np.nan
    
    # Esquema implícito
    T_imp = T_init.copy()
    for step in range(N_steps):
        t = step * dt
        T_imp = esquema_implicito(T_imp, r)
        T_imp[0] = T_media + A * np.sin(omega * (t + dt))
    T_ana = solucao_analitica(z, t_final, alpha, T_media, A, omega)
    erro_imp = np.mean(np.abs(T_imp - T_ana))
    
    # Crank-Nicolson
    T_cn = T_init.copy()
    for step in range(N_steps):
        t = step * dt
        T_cn = esquema_crank_nicolson(T_cn, r)
        T_cn[0] = T_media + A * np.sin(omega * (t + dt))
    erro_cn = np.mean(np.abs(T_cn - T_ana))
    
    resultados['dt'].append(dt)
    resultados['r'].append(r)
    resultados['erro_exp'].append(erro_exp)
    resultados['erro_imp'].append(erro_imp)
    resultados['erro_cn'].append(erro_cn)

# Tabela de resultados
print(f"\n{'Δt [s]':<10} {'r':<8} {'Erro Expl.':<12} {'Erro Impl.':<12} {'Erro CN':<12}")
print("-" * 60)
for i in range(len(dt_values)):
    exp_str = f"{resultados['erro_exp'][i]:.4f}" if not np.isnan(resultados['erro_exp'][i]) else "INSTÁVEL"
    print(f"{resultados['dt'][i]:<10} {resultados['r'][i]:<8.4f} {exp_str:<12} "
          f"{resultados['erro_imp'][i]:<12.4f} {resultados['erro_cn'][i]:<12.4f}")

print("\n💡 Interpretação:")
print("  • Explícito: mais preciso para r pequeno, mas instável para r > 0,5")
print("  • Implícito: menos preciso (erro de 1ª ordem no tempo)")
print("  • Crank-Nicolson: mais preciso (erro de 2ª ordem no tempo)")
print("  • Para r grande: CN é a melhor escolha (preciso e estável)")

---

## 💡 Seção 4: Discussão e Conclusões

### Resumo da Análise

| Esquema | Estabilidade | Precisão (tempo) | Custo | Recomendação |
|---------|--------------|------------------|-------|--------------|
| Explícito | Condicional ($r \leq 0,5$) | 1ª ordem | Baixo | $r$ pequeno |
| Implícito | Incondicional | 1ª ordem | Médio | $r$ grande |
| Crank-Nicolson | Incondicional | 2ª ordem | Médio | Geral |

### Recomendações Práticas

1. **Para $r \leq 0,5$:** Use esquema explícito (mais simples e rápido)
2. **Para $r > 0,5$:** Use Crank-Nicolson (preciso e estável)
3. **Para problemas não-lineares:** Use implícito com iteração de Picard
4. **Para malhas muito finas:** Implícito ou CN são essenciais

### Aplicações em Transferência de Calor em Solos

- **Ciclos diurnos:** $\Delta t = 300$ s, $r \approx 0,1$ → explícito OK
- **Ciclos anuais:** $\Delta t = 3600$ s, $r \approx 1,2$ → CN recomendado
- **Geotermia rasa:** $\Delta t$ grande → implícito essencial

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 8](../notebooks/08_Calor_Solos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 8.2**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>